In [ ]:
# If needed, run once and restart the kernel
!pip install -U scikit-learn umap-learn matplotlib seaborn pandas pyarrow tqdm

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

# Resolve paths (works if your notebook is in 4_Embedding_Extraction or project root)
DATA_CANDIDATES = [
    Path("../1_Data"),
    Path("1_Data"),
    Path("../../1_Data"),
]
for p in DATA_CANDIDATES:
    if p.exists():
        DATA_DIR = p
        break
else:
    raise FileNotFoundError("Could not find 1_Data directory.")

CSV_PATH = DATA_DIR / "preprocessed_data.csv"
NPZ_PATH = DATA_DIR / "hf_esm2_t30_150M_embeddings_vh_vl.npz"
VH_PARQ  = DATA_DIR / "hf_esm2_t30_150M_vh_embeddings.parquet"
VL_PARQ  = DATA_DIR / "hf_esm2_t30_150M_vl_embeddings.parquet"

print("Using data dir:", DATA_DIR.resolve())

In [ ]:
# Load HIC (for coloring)
df_meta = pd.read_csv(CSV_PATH)
assert "Clone name" in df_meta.columns and "HIC retention time (min)" in df_meta.columns
hic = df_meta["HIC retention time (min)"].values
clone_names = df_meta["Clone name"].values

def load_embeddings():
    if NPZ_PATH.exists():
        npz = np.load(NPZ_PATH, allow_pickle=True)
        vh = npz["vh_embeddings"]
        vl = npz["vl_embeddings"]
        names_from_npz = npz["clone_names"]
        # Align order with df_meta if needed
        if len(names_from_npz) == len(clone_names) and not np.array_equal(names_from_npz, clone_names):
            # Build an index map if orders differ (rare)
            name_to_idx = {n: i for i, n in enumerate(names_from_npz)}
            order = [name_to_idx[n] for n in clone_names]
            vh = vh[order]
            vl = vl[order]
        return vh, vl
    elif VH_PARQ.exists() and VL_PARQ.exists():
        vh_df = pd.read_parquet(VH_PARQ)
        vl_df = pd.read_parquet(VL_PARQ)
        # Expect a "Clone name" column + feature columns
        # Align to df_meta order:
        vh_df = vh_df.set_index("Clone name").loc[clone_names].reset_index()
        vl_df = vl_df.set_index("Clone name").loc[clone_names].reset_index()
        vh = vh_df.drop(columns=["Clone name"]).values
        vl = vl_df.drop(columns=["Clone name"]).values
        return vh, vl
    else:
        raise FileNotFoundError(
            "No embeddings found. Expected either NPZ or Parquet files in 1_Data."
        )

vh_emb, vl_emb = load_embeddings()
print("VH embeddings:", vh_emb.shape, "VL embeddings:", vl_emb.shape)

# Combined VH+VL features (simple concatenation)
combined_emb = np.concatenate([vh_emb, vl_emb], axis=1)
print("Combined embeddings:", combined_emb.shape)

In [ ]:
def do_pca(X, n_components=2, scale=True, random_state=42):
    """
    Returns: (coords ndarray (N,2), explained_variance_ratio array (2,))
    """
    X_proc = X
    if scale:
        X_proc = StandardScaler().fit_transform(X)
    pca = PCA(n_components=n_components, random_state=random_state)
    coords = pca.fit_transform(X_proc)
    return coords, pca.explained_variance_ratio_

def do_umap(X, n_components=2, scale=True, n_neighbors=15, min_dist=0.15, metric="euclidean", random_state=42):
    """
    Returns: coords ndarray (N,2)
    """
    X_proc = X
    if scale:
        X_proc = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
        random_state=random_state,
        verbose=False,
    )
    coords = reducer.fit_transform(X_proc)
    return coords

In [ ]:
def scatter_xy(xy, color, title, cmap="viridis", s=18, alpha=0.9, save_path=None):
    plt.figure(figsize=(6, 5))
    sc = plt.scatter(xy[:, 0], xy[:, 1], c=color, cmap=cmap, s=s, alpha=alpha)
    cbar = plt.colorbar(sc)
    cbar.set_label("HIC retention time (min)")
    plt.title(title)
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# --- PCA ---
vh_pca, vh_exp = do_pca(vh_emb, scale=True)
vl_pca, vl_exp = do_pca(vl_emb, scale=True)
comb_pca, comb_exp = do_pca(combined_emb, scale=True)

print(f"Explained variance (VH PCA): {vh_exp[0]:.2%}, {vh_exp[1]:.2%}")
print(f"Explained variance (VL PCA): {vl_exp[0]:.2%}, {vl_exp[1]:.2%}")
print(f"Explained variance (Combined PCA): {comb_exp[0]:.2%}, {comb_exp[1]:.2%}")

scatter_xy(vh_pca, hic, f"VH embeddings – PCA (2D)\nExpl.Var: {vh_exp[0]:.1%} + {vh_exp[1]:.1%}",
           save_path=DATA_DIR / "plot_vh_pca.png")
scatter_xy(vl_pca, hic, f"VL embeddings – PCA (2D)\nExpl.Var: {vl_exp[0]:.1%} + {vl_exp[1]:.1%}",
           save_path=DATA_DIR / "plot_vl_pca.png")
scatter_xy(comb_pca, hic, f"VH+VL embeddings – PCA (2D)\nExpl.Var: {comb_exp[0]:.1%} + {comb_exp[1]:.1%}",
           save_path=DATA_DIR / "plot_combined_pca.png")

# --- UMAP ---
# You can tweak n_neighbors/min_dist to control local vs global structure
vh_umap = do_umap(vh_emb, scale=True, n_neighbors=20, min_dist=0.1)
vl_umap = do_umap(vl_emb, scale=True, n_neighbors=20, min_dist=0.1)
comb_umap = do_umap(combined_emb, scale=True, n_neighbors=20, min_dist=0.1)

scatter_xy(vh_umap, hic, "VH embeddings – UMAP (2D)",
           save_path=DATA_DIR / "plot_vh_umap.png")
scatter_xy(vl_umap, hic, "VL embeddings – UMAP (2D)",
           save_path=DATA_DIR / "plot_vl_umap.png")
scatter_xy(comb_umap, hic, "VH+VL embeddings – UMAP (2D)",
           save_path=DATA_DIR / "plot_combined_umap.png")

In [ ]:
# Anova för VH

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
X_umap = vh_umap          # (N, 2)
hic_values = hic          # (N,)

In [ ]:
n_clusters = 2  # justera efter vad du ser i UMAP
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(X_umap)

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(
    x=X_umap[:,0], y=X_umap[:,1],
    hue=cluster_labels, palette="tab10",
    s=30
)
plt.title("UMAP-kluster (VH)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(title="Kluster")
plt.tight_layout()
plt.show()

In [ ]:
anova_df = pd.DataFrame({
    "Clone": clone_names,
    "Cluster": cluster_labels,
    "HIC": hic_values
})

anova_df.head()

In [ ]:
groups = [
    anova_df.loc[anova_df["Cluster"] == c, "HIC"].values
    for c in sorted(anova_df["Cluster"].unique())
]

F_stat, p_value = f_oneway(*groups)

print(f"ANOVA F-statistik: {F_stat:.3f}")
print(f"ANOVA p-värde:     {p_value:.3e}")

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(
    data=anova_df,
    x="Cluster",
    y="HIC",
    palette="Set2"
)
sns.stripplot(
    data=anova_df,
    x="Cluster",
    y="HIC",
    color="black",
    size=4,
    alpha=0.6
)
plt.title("HIC retention per UMAP-kluster (VH)")
plt.xlabel("Kluster")
plt.ylabel("HIC retention time (min)")
plt.tight_layout()
plt.show()

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=anova_df["HIC"],
    groups=anova_df["Cluster"],
    alpha=0.05
)

print(tukey)

In [ ]:
#Anova för VL

In [ ]:
X_umap = vl_umap          # (N, 4)
hic_values = hic          # (N,)

In [ ]:
n_clusters = 4  # justera efter vad du ser i UMAP
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(X_umap)

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(
    x=X_umap[:,0], y=X_umap[:,1],
    hue=cluster_labels, palette="tab10",
    s=30
)
plt.title("UMAP-kluster (VL)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(title="Kluster")
plt.tight_layout()
plt.show()

In [ ]:
anova_df = pd.DataFrame({
    "Clone": clone_names,
    "Cluster": cluster_labels,
    "HIC": hic_values
})

anova_df.head()

In [ ]:
groups = [
    anova_df.loc[anova_df["Cluster"] == c, "HIC"].values
    for c in sorted(anova_df["Cluster"].unique())
]

F_stat, p_value = f_oneway(*groups)

print(f"ANOVA F-statistik: {F_stat:.3f}")
print(f"ANOVA p-värde:     {p_value:.3e}")

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=anova_df["HIC"],
    groups=anova_df["Cluster"],
    alpha=0.05
)

print(tukey)

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(
    data=anova_df,
    x="Cluster",
    y="HIC",
    palette="Set2"
)
sns.stripplot(
    data=anova_df,
    x="Cluster",
    y="HIC",
    color="black",
    size=4,
    alpha=0.6
)
plt.title("HIC retention per UMAP-kluster (VL)")
plt.xlabel("Kluster")
plt.ylabel("HIC retention time (min)")
plt.tight_layout()
plt.show()